# LLM prompting exercise: Single-label classification of parliamentary sentences

Develop a prompt that classifies whether a sentence assigns the UK national government
responsibility for making housing affordable. Return exactly one label: `yes`, `no`, or
`unclear`. Use the codebook from the slides and decide how to handle sentences unrelated
to housing. Complete the `TODO` cells, starting with one sentence before processing more.


<!-- <br><a target="_blank" href="https://colab.research.google.com/github/haukelicht/advanced_text_analysis/blob/main/llm_inference_exercise_housing_classification.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> -->

<!-- **Note:** If running on Google Colab, make sure to use a GPU runtime (go to Runtime > Change runtime type, select "T4 GPU", and click save). See [this guide](https://github.com/haukelicht/advanced_text_analysis/blob/main/setup/setup_colab_runtime.md). -->

In [1]:
# check if on colab
COLAB=True
try:
    import google.colab
except:
    COLAB=False

if COLAB:
    # shallow clone of current state of main branch 
    !git clone --branch main --single-branch --depth 1 --filter=blob:none https://github.com/haukelicht/advanced_text_analysis.git
    # install required packages
    %pip install -q openai~=3.8.0

In [2]:
import os
import json
from pathlib import Path
import pandas as pd
from openai import OpenAI

In [3]:
hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Set HF_TOKEN in the project root .env file, then rerun setup.")

In [4]:
client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=hf_token,
    timeout=180,
    max_retries=0,
)

MODEL = "Qwen/Qwen2.5-72B-Instruct:deepinfra"

In [5]:
base_path = Path("/content/advanced_text_analysis/" if COLAB else "../../")
data_path = "data/labeled/sylvester_parlee_2022"
data_path = base_path / data_path
data_file = "sylvester_parlee_2022-uk_cap_sentences.csv"

In [6]:
SEED = 42

## Load and sample the ParlEE UK CAP data

We use a `cap_topic == 14` subset of the data
If necessary, the cell downloads and caches the CSV locally.

In [7]:
fp = data_path / data_file
if not fp.exists():
    url = "https://cta-text-datasets.s3.eu-central-1.amazonaws.com/labeled/sylvester_parlee_2022/sylvester_parlee_2022-uk_cap_sentences.csv"
    df = pd.read_csv(url)
    fp.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(fp, index=False)

# read the data
df = pd.read_csv(fp)

In [8]:
df = df.dropna(subset=["text", "cap_topic"]).copy()
df["text"] = df["text"].astype(str).str.strip()
df = df.loc[df["text"].ne("")].copy()
df["cap_housing_topic"] = df["cap_topic"].eq(14)

sample_df = (
    df.groupby("cap_housing_topic", group_keys=False)
    .sample(n=20, random_state=SEED)
    .sample(frac=1, random_state=SEED)
    .reset_index(drop=True)
)

# Keep CAP labels separate from the text sent to the model.
texts = sample_df[["text_id", "text"]].copy()
texts.head()

,text_id,text
0,1448025,What is important is the quality of the finish...
1,1789844,"For fear of being called to order by you, Mr S..."
2,174658,"Like the shadow Minister, I went out with a do..."
3,703974,"This magical piece of software, so much cheape..."
4,1969071,"The reports, particularly the Begg report, on ..."


::: {.callout-warning title="Topic codes are not responsibility labels"}

CAP topic 14 is **Community Development, Planning and Housing Issues**. It does not
indicate whether a sentence attributes responsibility to the national government.
Use the codes for sampling, not as reference answers for this task. This balanced
exercise sample also does not represent topic prevalence in the corpus.

:::

## Set up the API client

If not yet done, put the provided API key in `.env` in the project root:

```dotenv
HF_TOKEN=replace_with_the_teaching_key
```

Keep `.env` out of version control and do not print the key.
The client and model follow the API-basics notebook. The provider requires available
credit and access to the chosen model.

In [9]:
MODEL_ID = "Qwen/Qwen2.5-72B-Instruct"
PROVIDER = "deepinfra"
model_id = f"{MODEL_ID}:{PROVIDER}"

## 1. Write your prompt

Include the concept, category definitions, decision rules, input boundaries, and the
one-label output requirement. Keep the prompt version so you can compare revisions.

In [10]:
ALLOWED_LABELS = {"yes", "no", "unclear"}

# TODO: Replace the empty string with your coding instructions.
# Define all three labels, including what to do with non-housing sentences.
# Request only the selected label, with no explanation.
task_instruction = """
""".strip()

## 2. Construct the messages for one sentence

In [11]:
example = texts.iloc[0]
print(example["text"])

What is important is the quality of the finished product and whether it meets the specific needs of patients, not where or by whom the food is produced or prepared.


In [14]:
example_text = example['text']
text_template = 'Text: """{text}"""'
formatted_text = text_template.format(text=example_text)
print(formatted_text)

print(f'Text: """{example_text}"""')

Text: """What is important is the quality of the finished product and whether it meets the specific needs of patients, not where or by whom the food is produced or prepared."""
Text: """What is important is the quality of the finished product and whether it meets the specific needs of patients, not where or by whom the food is produced or prepared."""


In [ ]:
from pydantic import BaseModel
from typing import Literal, List

class Person(BaseModel):
    name: str
    office: str

class ExampleOutputClass(BaseModel):
    classification: Literal["yes", "no"]
    explanation: str
    phrase: List[str]
    persons: List[Person]

In [20]:
import json
type(json.loads('{"sentiment": "positive"}'))

dict

In [15]:
number = 1.23456
print(f'number = {number:.2f}')

number = 1.23


In [ ]:
def make_messages(text):
    # TODO: Return a list of role/content dictionaries.
    # Use a system message for task_instruction.
    # Use a user message for the delimited sentence (text).
    # Do not include CAP codes or previous model answers.
    raise NotImplementedError("Complete the messages list.")

assert task_instruction, "Write your prompt first."
messages = make_messages(example["text"])
messages

## 3. Send the request

In [ ]:
# TODO: Call client.chat_completion with MODEL_ID and messages.
# Supply max_tokens=32 and temperature=0.2, as in the API lesson.
# Replace None with the returned response object.
response = None

## 4. Extract and check the generated label

In [ ]:
def parse_response(response):
    # TODO: Extract the generated text from the first choice's message.
    # Keep that text as raw_label. Handle a missing content value.
    # Strip whitespace and lowercase the text to obtain label.
    # Check whether label belongs to ALLOWED_LABELS.
    # Return a dictionary with raw_label, label, and valid_label.
    raise NotImplementedError("Complete response parsing.")

assert response is not None, "Send your request first."
parsed = parse_response(response)
parsed

::: {.callout-tip title="Inspect one result before continuing"}

Read the sentence alongside the response. Is the label allowed, and does it follow your
coding rule? Check the response's finish reason for truncation. An allowed label can
still be substantively wrong.

:::

## 5. Apply your prompt to the sample

In [ ]:
# TODO: Complete the loop after the one-sentence example works.
records = []
for row in texts.itertuples(index=False):
    # 1. Build fresh messages with make_messages(row.text).
    # 2. Send a completion request using the same generation settings.
    # 3. Parse it with parse_response(...).
    # 4. Append a record containing sentence_id, text, and parsed fields,
    #    plus MODEL_ID, PROVIDER, and PROMPT_VERSION.
    # Keep failed requests identifiable by sentence_id.
    raise NotImplementedError("Complete the classification loop.")

results = pd.DataFrame(records)
results

## 6. Review and save

Read several labels, including unclear or invalid outputs. Note one codebook or prompt
revision you would make. Use human judgments, rather than the CAP topic indicator, to
assess the responsibility classifications.

In [ ]:
# TODO: Inspect selected sentences alongside their model labels.
# TODO: Save results and task_instruction under a versioned filename
#       in PROJECT_ROOT / "outputs". Keep the sentence IDs.